# EDA: feature by feature

For every feature, three questions:
1. **How many values are missing?** Is the fraud rate different when the value is missing?
2. **Which values or groups have more frauds?** (blue = purchases, pink = % of frauds, dashed line = average)
3. **In one sentence: is this feature useful?** → the *Finding* under each chart.

How strong is a feature: riskiest group ÷ average fraud rate. **Under 1.5 weak, over 2 interesting, over 3 strong.**

Types of feature:
- **numbers** (the double makes sense: amount, days, counts) → `numeric(train, col)`: 10 groups with the same number of purchases
- **codes / categories** (a label, like a postal code: card, addr, email, M) → `category(train, col)`: most frequent values
- **many similar columns** (C, D, V) → `overview(train, cols)` (one row per column) and `copies(train, cols)` (which columns are almost copies)

Only months 0–3: the last months are kept hidden for the final test of the model.

## Functions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BLUE = "#A7C7E7"     # pastel blue: purchases
PINK = "#F4A7B9"     # pastel pink: frauds
LINE = "#6B6B6B"     # dark grey: average line
IMAGES = Path("../docs/images")   # charts saved here are shown in the README


def missing(df, col):
    """Question 1: how many values are missing, and the fraud rate with / without a value."""
    m = df[col].isna()
    print(f"{col}: missing {100 * m.mean():.1f}% of the rows")
    if 0 < m.sum() < len(df):
        print(f"  fraud rate WITH a value:    {100 * df.loc[~m, 'isFraud'].mean():.2f}%")
        print(f"  fraud rate WITHOUT a value: {100 * df.loc[m, 'isFraud'].mean():.2f}%")


def _plot(groups, overall_pct, title, save=None):
    """groups must have 'purchases' and 'fraud_pct' (already in %). save = file name in docs/images/."""
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    x = range(len(groups))
    top.bar(x, groups["purchases"], color=BLUE)
    top.set_ylabel("purchases")
    top.set_title(title)
    bottom.bar(x, groups["fraud_pct"], color=PINK)
    bottom.axhline(overall_pct, color=LINE, linestyle="--", label=f"average {overall_pct:.1f}%")
    bottom.set_ylabel("frauds (%)")
    bottom.legend(frameon=False)
    bottom.set_xticks(list(x), list(groups.index), rotation=45, ha="right")
    for ax in (top, bottom):
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    if save:
        IMAGES.mkdir(parents=True, exist_ok=True)
        plt.savefig(IMAGES / save, dpi=120)
    plt.show()


def _summary(df, labels):
    """Purchases and % of frauds for each group."""
    groups = df.groupby(labels)["isFraud"].agg(purchases="size", fraud_pct="mean")
    groups["fraud_pct"] = groups["fraud_pct"] * 100          # to % (ONLY here)
    return groups


def numeric(df, col, n_groups=10, save=None):
    """Numbers: 10 groups with the same number of purchases, from the lowest to the highest values."""
    missing(df, col)
    g = pd.qcut(df[col], n_groups, duplicates="drop")
    names = {c: f"{c.left:,.0f}–{c.right:,.0f}" for c in g.cat.categories}
    labels = g.map(names).astype(str).where(df[col].notna(), "missing")
    groups = _summary(df, labels)
    order = list(names.values()) + ["missing"]
    groups = groups.reindex([o for o in order if o in groups.index])
    _plot(groups, df["isFraud"].mean() * 100, f"{col}: purchases and frauds per group", save)
    return groups


def category(df, col, top=15, save=None):
    """Codes: the `top` most frequent values, the others in "other", empty values in "missing"."""
    missing(df, col)
    v = df[col].astype("object")
    frequent = v.value_counts().head(top).index
    labels = v.where(v.isin(frequent), "other").where(v.notna(), "missing").astype(str)
    groups = _summary(df, labels).sort_values("purchases", ascending=False)
    _plot(groups, df["isFraud"].mean() * 100, f"{col}: purchases and frauds per value", save)
    return groups


def by_time(df, col, title, save=None):
    """Time columns (month, hour, weekday): one bar per value, in order."""
    groups = _summary(df, df[col]).sort_index()
    _plot(groups, df["isFraud"].mean() * 100, title, save)
    return groups


def overview(df, cols):
    """One row per column: missing values, fraud when missing, weakest and strongest group, strength."""
    avg = df["isFraud"].mean()
    rows = []
    for c in cols:
        values, m = df[c], df[c].isna()
        groups = pd.qcut(values, 10, duplicates="drop")
        rates = df.groupby(groups, observed=True)["isFraud"].mean()
        rows.append({
            "column": c,
            "missing %": round(100 * m.mean(), 1),
            "fraud % if missing": round(100 * df.loc[m, "isFraud"].mean(), 1) if m.any() else None,
            "lowest group %": round(100 * rates.min(), 1),
            "highest group %": round(100 * rates.max(), 1),
            "strength": round(rates.max() / avg, 1),       # highest group / average
        })
    return pd.DataFrame(rows).set_index("column")


def copies(df, cols, limit=0.95):
    """Heatmap of the correlation between columns; prints the pairs that are almost copies (>= limit)."""
    corr = df[cols].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(corr, cmap="PuBu", vmin=0, vmax=1)          # white = unrelated, blue = copies
    ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")
    ax.set_yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title("How much the columns move together (1 = copies)")
    plt.tight_layout()
    plt.show()
    pairs = [(a, b, round(float(corr.loc[a, b]), 2)) for i, a in enumerate(cols) for b in cols[i + 1:]
             if corr.loc[a, b] >= limit]
    print("almost copies:", pairs if pairs else "none")


def keep_one(df, cols, limit=0.95):
    """Groups of copies: columns linked by a chain of correlations >= limit (if A~B and B~C, then A, B, C
    are one group). From each group keep ONE column: the one with fewer missing values; if equal, the one
    with higher strength. Returns the list of kept columns."""
    corr = df[cols].corr().abs()
    parent = {c: c for c in cols}
    def root(c):
        while parent[c] != c:
            c = parent[c]
        return c
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            if corr.loc[a, b] >= limit:
                parent[root(b)] = root(a)
    groups = {}
    for c in cols:
        groups.setdefault(root(c), []).append(c)
    table = overview(df, cols)
    kept = []
    for members in groups.values():
        best = min(members, key=lambda c: (table.loc[c, "missing %"], -table.loc[c, "strength"]))
        kept.append(best)
        if len(members) > 1:
            print(f"copies {members} -> keep {best}")
    print(f"{len(cols)} columns -> {len(kept)} kept: {kept}")
    return kept

## Data

In [ ]:
DATA_FILE = "../data/raw/train_transaction.csv"   # if it is not found, write the full path here

df = pd.read_csv(DATA_FILE)

DAY, HOUR = 86400, 3600                         # TransactionDT is in seconds from a hidden start
df["day"] = df["TransactionDT"] // DAY
df["month"] = df["day"] // 30
df.loc[df["month"] == 6, "month"] = 5          # "month 6" = last 2 days, joined to month 5
df["hour"] = (df["TransactionDT"] // HOUR) % 24
df["weekday"] = df["day"] % 7
df = df.copy()

print("all data:", df.shape, f"fraud rate {df['isFraud'].mean() * 100:.2f}%")
train = df[df["month"] <= 3].copy()           # EDA only on months 0-3
print("months 0-3:", train.shape, f"fraud rate {train['isFraud'].mean() * 100:.2f}%")

## 0. Time

In [ ]:
by_time(df, "month", "Purchases and frauds per month", save="per_month.png")   # all months: only to see if fraud changes over time

**Finding:** The fraud rate changes over time: 2.5% in month 0 (probably the Christmas period, many honest purchases), then 3.4–4%. The model must be trained on the past and tested on the future, never on a random split.

In [ ]:
by_time(train, "hour", "Purchases and frauds per hour of the day", save="per_hour.png")

**Finding:** Strong. Between hours 5 and 10 (the night: very few purchases) the fraud rate goes up to 10.6%, 3x the average. Most frauds still happen during the day, when there are many more purchases.

In [ ]:
by_time(train, "weekday", "Purchases and frauds per day of the week", save="per_weekday.png")

**Finding:** _write here._

## 1. TransactionAmt (amount)

Type: **number**

In [ ]:
numeric(train, "TransactionAmt")

**Finding:** Weak alone: the fraud rate goes from 2% to 5.5% (average 3.5%). U-shape: more fraud for very small amounts (possibly thieves testing the card) and very large amounts. More useful compared to the card's usual amount → new feature: amount / average amount of the card.

## 2. ProductCD (type of product)

Type: **code, few values**

In [ ]:
category(train, "ProductCD")

**Finding:** Strong. Type C has 11% fraud (3x the average) on ~50,000 purchases: almost as many frauds as type W, which has 6x more purchases. W, the most common type, is the safest (2%).

## 3. card1 – card6 (the card)

Type: **codes**. card4 and card6 have few values, card1/2/3/5 many.

In [ ]:
category(train, "card4")

**Finding:** Weak. Visa and Mastercard are at the average; American Express is below it (2.8%). Only Discover is riskier (6.6%), on a few thousand purchases. Missing values: below the average.

In [ ]:
category(train, "card6")

**Finding:** Useful. Credit cards have almost 3x the fraud rate of debit cards (6.6% vs 2.4%), on ~110,000 purchases.

In [ ]:
category(train, "card1")

**Finding:** Too many values (thousands) to use as a category: each code is a small group of cards. Some codes are very risky (9633: 16.5% on ~3,000 purchases). Main use: one of the pieces to identify the same card / customer.

In [ ]:
category(train, "card2")

**Finding:** Useful but weaker. A few codes are about 2x the average (545, 408: ~7.5%). When missing, fraud is 5.9%.

In [ ]:
category(train, "card3")

**Finding:** Strong. One value (150, probably the home country) covers most purchases and is below the average (2.5%). Value 185 (probably foreign cards) has 12.4% fraud on ~40,000 purchases (3.5x the average).

In [ ]:
category(train, "card5")

**Finding:** Useful. Some card groups (bank or category) are much riskier: 137 has 14% fraud on ~8,000 purchases, 138 has 8%. The largest group (226) is at the average.

## 4. Address and distance

`addr1`, `addr2`: codes (billing region and country) · `dist1`, `dist2`: numbers

In [ ]:
category(train, "addr1")

**Finding:** The value itself says little (all regions are around or below the average). What matters is when it is missing: 11% fraud on ~47,000 purchases (3x the average). No billing address = no address check = easier for a thief → new feature: addr1 missing yes/no.

In [ ]:
category(train, "addr2")

**Finding:** One country (87) covers almost everything, at 2%. Missing: 11% (same rows as addr1). Value 65 has 58% fraud but on very few purchases: to watch, not to trust.

In [ ]:
numeric(train, "dist1")

**Finding:** Weak but ordered: the larger the distance, the more fraud (from 1.3% to 3.1%). Missing: 4.4%, a bit above the average.

In [ ]:
numeric(train, "dist2")

**Finding:** What matters is that it exists. It is present in only ~7% of purchases, and when present fraud is 6-13% whatever the distance; when missing, 3% → new feature: dist2 present yes/no.

## 5. Email domains

Type: **codes, many values**

In [ ]:
category(train, "P_emaildomain")

**Finding:** Useful. Some domains are riskier: outlook.com 9% (few purchases), hotmail.com 5.3%, gmail.com 4.4% (on 158,000 purchases). Internet provider emails (att.net, sbcglobal.net, verizon.net) are very safe (under 0.5%).

In [ ]:
category(train, "R_emaildomain")

**Finding:** What matters is that it exists: with a receiver email, fraud is much higher (gmail 11%, outlook 15%, icloud 11.5%); without it, 2%. Buying for someone else is a way for a thief to receive the goods → new feature: R_emaildomain present yes/no.

## 6. C1 – C14 (counts)

Type: **numbers**. First C1 alone, then all the C together: a table (one row per column) and the copies.

In [ ]:
numeric(train, "C1")

**Finding:** Strong and ordered: the higher the count (e.g. how many addresses are linked to the card), the more fraud, from 2.4% to 8.6%.

In [ ]:
C = [f"C{i}" for i in range(1, 15)]
overview(train, C)

**Finding:** _write here._ Which C are strong (strength over 2)?

In [ ]:
copies(train, C)

**Finding:** Ten C are almost copies of each other (C1, C2, C4, C6, C7, C8, C10, C11, C12, C14: correlation 0.95–1.0, linked in one chain). C3, C5, C9 and C13 are not copies of any other column.


In [ ]:
C_keep = keep_one(train, C)

**Finding:** From 14 C to 5: C7 (for the group of ten copies), C3, C5, C9, C13.

## 7. D1 – D15 (days passed)

Type: **numbers**. Same method as the C.

In [ ]:
numeric(train, "D1")

**Finding:** Strong and ordered: cards seen recently are riskier (4.6% for 0-21 days vs 1.3% for more than one year). But an old card can be stolen too: D1 alone is not enough, it must be combined with "is this purchase different from the card's usual behaviour?".

In [ ]:
D = [f"D{i}" for i in range(1, 16)]
overview(train, D)

**Finding:** _write here._ Which D are strong? Which are almost always missing?

In [ ]:
copies(train, D)

**Finding:** Three groups of copies: {D1, D2}, {D4, D6, D12}, {D5, D7}. The other D are not copies of any other column.

In [ ]:
D_keep = keep_one(train, D)

**Finding:** From 15 D to 11: D1 (for D1/D2), D4 (for D4/D6/D12), D5 (for D5/D7), and D3, D8, D9, D10, D11, D13, D14, D15.

## 8. M1 – M9 (matches, true / false)

Type: **codes, few values** (T = true, F = false; M4 has M0/M1/M2).

In [ ]:
for col in [f"M{i}" for i in range(1, 10)]:
    category(train, col)

**Finding:** _write here._

## 9. V1 – V339

339 columns. Two steps:
1. group the V by **missing pattern**: columns that are empty on exactly the same rows form a block;
2. inside each block, the same method as the C: `overview` and `copies`, keep one column per group of copies.

### Step 1: blocks of V that are missing on the same rows

In [ ]:
V = [f"V{i}" for i in range(1, 340)]
missing_count = train[V].isna().sum()                     # how many empty rows each V has
blocks = [list(cols) for cols in missing_count.groupby(missing_count).groups.values()]

pd.DataFrame({
    "block": range(len(blocks)),
    "columns": [len(b) for b in blocks],
    "missing %": [round(100 * missing_count[b[0]] / len(train), 1) for b in blocks],
    "from": [b[0] for b in blocks],
    "to": [b[-1] for b in blocks],
}).set_index("block")

**Finding:** _write here: how many blocks, how big, how empty._

### Step 2: inside each block, keep one column per group of copies

For C and D the limit is 0.95 ("almost identical"). For the V it is **0.75** ("very similar"): with 0.95 the V went only from 339 to ~290, because many V are variants of the same count built by Vesta, very similar but not identical. The V are anonymous, so we lose nothing by keeping one per group of similar columns.

In [ ]:
V_LIMIT = 0.75      # V: 'very similar' is enough (they are anonymous, we don't explain them one by one)
V_keep = []
for n, cols in enumerate(blocks):
    print(f"\n--- block {n} ({len(cols)} columns) ---")
    V_keep += keep_one(train, cols, limit=V_LIMIT)
print(f"\nV: {len(V)} columns -> {len(V_keep)} kept")

### Step 3: which of the kept V are strong

In [ ]:
overview(train, V_keep).sort_values("strength", ascending=False).head(20)

**Finding:** _write here: how many V are left, and which are the strongest._

## 10. Identity table (device, browser)

To be joined with `train_transaction` on `TransactionID`. Later.

## 11. Selected features

What goes to the next step (feature engineering and model). Kept:
- the readable features found useful above;
- from C, D and V: one column per group of copies (`C_keep`, `D_keep`, `V_keep`);
- all the M (few columns, to be confirmed by their charts).

New features to **build** in the next step (they are not columns yet): `addr1_missing`, `dist2_present`,
`R_emaildomain_present`, amount compared to the card's usual amount. `card1` is kept to rebuild the card /
customer. `month` and `day` are not features: they are only used to split the data in time.

In [ ]:
import json

READABLE = ["TransactionAmt", "ProductCD", "hour", "weekday",
            "card1", "card2", "card3", "card4", "card5", "card6",
            "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain"]
M = [f"M{i}" for i in range(1, 10)]

selected = READABLE + C_keep + D_keep + M + V_keep
print(f"{len(selected)} features kept, out of {df.shape[1]} columns")
print(f"  readable {len(READABLE)} | C {len(C_keep)} | D {len(D_keep)} | M {len(M)} | V {len(V_keep)}")

with open("eda_selected_features.json", "w") as f:        # read by the next notebook
    json.dump(selected, f, indent=1)

## Statistics concepts used in this analysis

| concept | where |
|---|---|
| **Base rate and class imbalance**: only 3.5% of purchases are frauds, so accuracy is useless | the first numbers |
| **Conditional probability**: fraud rate *given* a group, P(fraud \| group) | every pink bar |
| **Lift (relative risk)**: group rate ÷ average rate, our "strength" | ProductCD, card3, C, D |
| **Sample size and noise**: a % on a small group can be chance | card4 Discover, addr2 = 65 |
| **Quantile binning**: groups with the same number of purchases (`qcut`) | every number |
| **Non-linear relationships**: correlation misses mountains and U-shapes | hour, amount |
| **Variable types**: numbers vs codes (nominal categories), high cardinality | card1, addr1 |
| **Informative missing values** (not missing at random) | addr1, dist2, R_emaildomain |
| **Multicollinearity**: columns that are almost copies, one kept per group | C, D, V |
| **Pearson correlation is sensitive to extreme values** | C (values up to thousands) |
| **Non-stationarity / drift**: the fraud rate changes over time | per month |
| **Data leakage and time split**: decisions only on months 0-3 | the whole analysis |

## Summary

| feature | useful? | what we do with it |
|---|---|---|
| hour | strong (3x at night) | feature |
| TransactionAmt | weak alone | new feature: amount / card's usual amount |
| ProductCD | strong | category; rule candidate (ProductCD = C) |
| card4 | weak | category (only Discover stands out) |
| card6 | useful | category |
| card1 | too many values | used to identify the card / customer |
| card2 | useful, weaker | category + missing flag |
| card3 | strong | category (185) |
| card5 | useful | category |
| addr1 / addr2 | the missing value is strong | missing flag |
| dist1 | weak | number |
| dist2 | presence is strong | present flag |
| P_emaildomain | useful | category (clean similar domains) |
| R_emaildomain | presence is strong | present flag + category |
| C1 | strong | number |
| D1 | strong | number |
| C2 – C14 | | |
| D2 – D15 | | |
| M1 – M9 | | |